In [ ]:
!git clone https://github.com/rawaf086-lab/fashion-mnist-occlusion-robustness.git
%cd fashion-mnist-occlusion-robustness
!pip install -q -r Dependency_List.txt
!mv results reference_results
!mkdir results
!cp reference_results/training_history.csv results/

In [ ]:
import os
import pandas as pd
from huggingface_hub import hf_hub_download

dataset_path = "."
os.makedirs(os.path.join(dataset_path, "results"), exist_ok=True)

REPO_ID = "zalando-datasets/fashion_mnist"
train_path = hf_hub_download(REPO_ID, "fashion_mnist/train-00000-of-00001.parquet",
                             repo_type="dataset", local_dir=dataset_path)
test_path = hf_hub_download(REPO_ID, "fashion_mnist/test-00000-of-00001.parquet",
                            repo_type="dataset", local_dir=dataset_path)

train_df = pd.read_parquet(train_path)
test_df = pd.read_parquet(test_path)

print("Training images:", len(train_df))
print("Testing images:", len(test_df))
print("\nColumns:", train_df.columns.tolist())

In [ ]:
from sklearn.model_selection import train_test_split

SEED = 42

train_data, val_data = train_test_split(
    train_df,
    test_size=0.10,
    random_state=SEED,
    stratify=train_df["label"]
)

train_data = train_data.reset_index(drop=True)
val_data = val_data.reset_index(drop=True)

print("Training:", len(train_data))
print("Validation:", len(val_data))
print("Testing:", len(test_df))

print("\nTraining class distribution:")
print(train_data["label"].value_counts().sort_index())

print("\nValidation class distribution:")
print(val_data["label"].value_counts().sort_index())

In [ ]:
# Cell 5: Display Sample Images and Verify Image Dimensions
from PIL import Image
from io import BytesIO
import matplotlib.pyplot as plt

classes = [
    "T-shirt/top", "Trouser", "Pullover", "Dress",
    "Coat", "Sandal", "Shirt", "Sneaker",
    "Bag", "Ankle boot"
]

fig, axes = plt.subplots(2, 5, figsize=(12, 5))

for label, ax in enumerate(axes.flat):
    sample = train_data[train_data["label"] == label].iloc[0]

    img = Image.open(BytesIO(sample["image"]["bytes"]))

    ax.imshow(img, cmap="gray")
    ax.set_title(classes[label])
    ax.axis("off")

plt.tight_layout()
plt.show()

print("Image size:", img.size)
print("Image mode:", img.mode)

In [ ]:
# Cell 6: Convert Image to Tensor and Normalize Pixel Values

import torch
from torchvision import transforms
from PIL import Image
from io import BytesIO

# Define the transformation
transform = transforms.ToTensor()

# Select a sample image from training data
sample = train_data.iloc[0]
img = Image.open(BytesIO(sample["image"]["bytes"])).convert("L")

# Apply transformation
img_tensor = transform(img)

# Verify the result
print("Tensor shape:", img_tensor.shape)
print("Data type:", img_tensor.dtype)
print("Minimum pixel value:", img_tensor.min().item())
print("Maximum pixel value:", img_tensor.max().item())

In [ ]:
# Cell 7: Set Random Seeds and Check Computing Device

import random
import numpy as np
import torch

# Set random seed for reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Configure reproducible CUDA behavior
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Select GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Random Seed:", SEED)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Cell 8: Create PyTorch Datasets and DataLoaders

from torch.utils.data import Dataset, DataLoader
from PIL import Image
from io import BytesIO

# Define a custom dataset
class FashionDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.data = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data.iloc[idx]

        image = Image.open(
            BytesIO(sample["image"]["bytes"])
        ).convert("L")

        label = int(sample["label"])

        if self.transform:
            image = self.transform(image)

        return image, label


# Create datasets
train_dataset = FashionDataset(train_data, transform)
val_dataset = FashionDataset(val_data, transform)
test_dataset = FashionDataset(test_df, transform)

# Create DataLoaders
batch_size = 128

train_loader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True
)

val_loader = DataLoader(
    val_dataset, batch_size=batch_size, shuffle=False
)

test_loader = DataLoader(
    test_dataset, batch_size=batch_size, shuffle=False
)

# Verify one training batch
images, labels = next(iter(train_loader))

print("Images shape:", images.shape)
print("Labels shape:", labels.shape)
print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Testing samples:", len(test_dataset))

In [ ]:
# Cell 9: Define the CNN Model Architecture

import torch
import torch.nn as nn
import torch.nn.functional as F

class FashionCNN(nn.Module):
    def __init__(self):
        super(FashionCNN, self).__init__()

        # First Convolutional Layer
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)

        # Second Convolutional Layer
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)

        # Max Pooling Layer
        self.pool = nn.MaxPool2d(2, 2)

        # Fully Connected Layers
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):

        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))

        x = torch.flatten(x, 1)

        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x


# Create the model and move it to GPU
model = FashionCNN().to(device)

# Display model architecture
print(model)

# Verify output dimensions
with torch.no_grad():
    sample_input = torch.zeros(1, 1, 28, 28).to(device)
    sample_output = model(sample_input)

print("\nOutput shape:", sample_output.shape)

In [ ]:
# Cell 10: Configure CNN Training Parameters

import torch.optim as optim

# Define the loss function
criterion = nn.CrossEntropyLoss()

# Define the optimizer
learning_rate = 0.001
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Set training parameters
num_epochs = 10

# Display configuration
print("Loss Function:", criterion)
print("Optimizer:", optimizer.__class__.__name__)
print("Learning Rate:", learning_rate)
print("Batch Size:", batch_size)
print("Number of Epochs:", num_epochs)
print("Device:", device)

In [ ]:
# Cell 11: Define CNN Training Function Without Automatic Execution

import os
import time
import torch

def train_cnn(model, train_loader, val_loader,
              criterion, optimizer, num_epochs,
              device, save_path):

    # Prevent accidental overwriting of a saved model
    if os.path.exists(save_path):
        raise FileExistsError(
            "Model file already exists. Use a new save path."
        )

    history = {
        "train_loss": [],
        "train_accuracy": [],
        "val_loss": [],
        "val_accuracy": []
    }

    best_val_accuracy = -1.0
    start_time = time.time()

    for epoch in range(num_epochs):

        # Training phase
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            train_loss += loss.item() * len(labels)
            train_correct += (
                outputs.argmax(1) == labels
            ).sum().item()
            train_total += len(labels)

        # Validation phase
        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * len(labels)
                val_correct += (
                    outputs.argmax(1) == labels
                ).sum().item()
                val_total += len(labels)

        train_loss /= train_total
        val_loss /= val_total

        train_accuracy = 100 * train_correct / train_total
        val_accuracy = 100 * val_correct / val_total

        # Store training history
        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_accuracy)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_accuracy)

        # Save the best model
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            torch.save(model.state_dict(), save_path)

        print(
            f"Epoch [{epoch+1}/{num_epochs}] | "
            f"Train Acc: {train_accuracy:.2f}% | "
            f"Val Acc: {val_accuracy:.2f}%"
        )

    training_time = time.time() - start_time

    return history, best_val_accuracy, training_time


RUN_TRAINING = False   # False = use released weights; True = retrain from scratch

if RUN_TRAINING:
    retrain_path = os.path.join(dataset_path, "retrained_cnn.pth")
    hist, best_val, secs = train_cnn(model, train_loader, val_loader, criterion,
                                     optimizer, num_epochs, device, retrain_path)
    h = pd.DataFrame(hist)
    h.insert(0, "epoch", range(1, len(h) + 1))
    h.to_csv(os.path.join(dataset_path, "results", "retrained_history.csv"), index=False)
    print(f"Best val acc: {best_val:.2f}% | time: {secs:.0f}s")

    m2 = FashionCNN().to(device)
    m2.load_state_dict(torch.load(retrain_path, map_location=device, weights_only=True))
    m2.eval()
    correct = 0
    with torch.inference_mode():
        for x, y in test_loader:
            correct += (m2(x.to(device)).argmax(1).cpu() == y).sum().item()
    print(f"Retrained test accuracy: {100 * correct / len(test_dataset):.2f}%")
else:
    print("Using released weights (no retraining).")

In [ ]:
# Cell 12: Restore Saved CNN Model Without Retraining

import os
import torch

model_path = os.path.join(dataset_path, "best_fashion_cnn.pth")

assert os.path.exists(model_path), "Saved model not found!"

best_model = FashionCNN().to(device)

best_model.load_state_dict(
    torch.load(model_path, map_location=device, weights_only=True)
)

best_model.eval()

print("Saved CNN model loaded successfully!")
print("Device:", device)
print("No retraining performed.")

In [ ]:
# Cell 13: Evaluate Loaded CNN and Verify Baseline Results

import numpy as np
import pandas as pd
import os
import torch

# Use the trained model loaded in Cell 11A
best_model.eval()

true_labels = []
pred_labels = []

# Evaluate original test images without training
with torch.inference_mode():
    for images, labels in test_loader:

        images = images.to(device)

        outputs = best_model(images)
        predictions = outputs.argmax(dim=1)

        true_labels.extend(labels.tolist())
        pred_labels.extend(predictions.cpu().tolist())

true_labels = np.array(true_labels)
pred_labels = np.array(pred_labels)

# Calculate overall baseline accuracy
baseline_accuracy = (
    (true_labels == pred_labels).mean() * 100
)

print(f"Baseline Test Accuracy: {baseline_accuracy:.2f}%")

# Calculate accuracy for each clothing class
print("\nPer-Class Accuracy:")

for i, class_name in enumerate(classes):

    mask = (true_labels == i)

    class_accuracy = (
        (pred_labels[mask] == true_labels[mask]).mean() * 100
    )

    print(f"{class_name}: {class_accuracy:.2f}%")

# Prepare baseline results
results_dir = os.path.join(dataset_path, "results")
os.makedirs(results_dir, exist_ok=True)

baseline_df = pd.DataFrame({
    "image_index": np.arange(len(true_labels)),
    "true_label": true_labels,
    "original_prediction": pred_labels
})

baseline_csv = os.path.join(
    results_dir, "baseline_predictions.csv"
)

# Verify previous results before saving anything
if os.path.exists(baseline_csv):

    previous = pd.read_csv(baseline_csv)

    assert previous.equals(baseline_df), (
        "Baseline predictions differ from saved results!"
    )

    print("\nPrevious baseline results verified unchanged.")

else:
    baseline_df.to_csv(baseline_csv, index=False)
    print("\nBaseline results saved.")

print("No retraining performed.")

In [ ]:
# Cell 14: Load and Display Saved Training History

import pandas as pd
import os

# Load previously saved training results
history_path = os.path.join(
    dataset_path, "results", "training_history.csv"
)

history_df = pd.read_csv(history_path)

# Display all 10 epochs
print("Saved Training History:")
display(history_df.round(4))

print("\nTotal Epochs:", len(history_df))
print(
    "Best Validation Accuracy:",
    f"{history_df['val_accuracy'].max():.2f}%"
)

print("No retraining performed.")

In [ ]:
# Cell 15: Define Final Foreground-Aware Occlusion Function

import torch

# Define occlusion mask sizes
occlusion_sizes = {
    "Small": 4,
    "Medium": 8,
    "Large": 12
}

# Define occlusion positions
occlusion_positions = {
    "Top": 0.2,
    "Center": 0.5,
    "Bottom": 0.8
}

# Define foreground threshold
foreground_threshold = 0.1


def apply_occlusion(image, position, size):

    # Preserve the original image
    occluded = image.clone()

    # Identify foreground pixels
    foreground = image[0] > foreground_threshold
    ys, xs = torch.where(foreground)

    if len(xs) == 0:
        raise ValueError("No foreground pixels detected.")

    # Find foreground boundaries
    y_min, y_max = ys.min().item(), ys.max().item()
    x_min, x_max = xs.min().item(), xs.max().item()

    # Determine target position
    target_y = y_min + occlusion_positions[position] * (y_max - y_min)
    target_x = (x_min + x_max) / 2

    # Find nearest foreground pixels
    y_distance = torch.abs(ys.float() - target_y)
    candidates = torch.where(y_distance == y_distance.min())[0]

    # Select foreground pixel closest to horizontal center
    x_distance = torch.abs(xs[candidates].float() - target_x)
    selected = candidates[torch.argmin(x_distance)]

    center_y = ys[selected].item()
    center_x = xs[selected].item()

    # Keep mask within image boundaries
    height, width = image.shape[1:]

    x_start = max(0, min(center_x - size // 2, width - size))
    y_start = max(0, min(center_y - size // 2, height - size))

    # Calculate hidden foreground percentage
    hidden_pixels = foreground[
        y_start:y_start + size,
        x_start:x_start + size
    ].sum().item()

    total_foreground = foreground.sum().item()

    percentage = 100 * hidden_pixels / total_foreground

    # Apply black occlusion mask
    occluded[
        :,
        y_start:y_start + size,
        x_start:x_start + size
    ] = 0.0

    return occluded, percentage


print("Final Foreground-Aware Occlusion Function Ready!")

In [ ]:
# Cell 16: Visualize Original Image and 9 Occlusion Conditions

import matplotlib.pyplot as plt
import torch

# Select a T-shirt image from the Validation Dataset
# Use positional indexing to avoid index mismatch
sample_idx = int(
    (val_data["label"] == 0).to_numpy().nonzero()[0][0]
)

image, label = val_dataset[sample_idx]

# Verify that the selected image is a T-shirt
assert int(label) == 0, "Selected image is not a T-shirt."

# Keep a copy to verify the original is unchanged
original_image = image.clone()

# Create a figure with 10 images
fig, axes = plt.subplots(2, 5, figsize=(15, 7))
axes = axes.flatten()

# Display the original image
axes[0].imshow(
    image.squeeze(),
    cmap="gray",
    vmin=0,
    vmax=1
)
axes[0].set_title("Original Image")
axes[0].axis("off")

# Apply all 9 occlusion combinations
idx = 1

for position in occlusion_positions:
    for size_name, size_value in occlusion_sizes.items():

        occluded, percentage = apply_occlusion(
            image, position, size_value
        )

        # Verify output validity
        assert occluded.shape == image.shape
        assert 0 <= percentage <= 100

        axes[idx].imshow(
            occluded.squeeze(),
            cmap="gray",
            vmin=0,
            vmax=1
        )

        axes[idx].set_title(
            f"{position} - {size_name}\n"
            f"Hidden Foreground: {percentage:.1f}%"
        )

        axes[idx].axis("off")
        idx += 1

plt.suptitle(
    f"Foreground-Aware Occlusion: {classes[label]}",
    fontsize=15
)

plt.tight_layout(rect=[0, 0, 1, 0.94], h_pad=3.0)
plt.show()

# Verify that occlusion did not modify the original image
print(
    "Original image unchanged:",
    torch.equal(image, original_image)
)

print("Occlusion conditions tested:", idx - 1)

In [ ]:
# Cell 17: Validate Occlusion Across All 10 Clothing Classes

import matplotlib.pyplot as plt
import torch

# Create a figure for all clothing classes
fig, axes = plt.subplots(10, 4, figsize=(13, 26))

verified_cases = 0

for class_idx, class_name in enumerate(classes):

    # Select one validation image from each class
    # Use positional indexing to avoid index mismatch
    sample_idx = int(
        (val_data["label"] == class_idx)
        .to_numpy()
        .nonzero()[0][0]
    )

    image, label = val_dataset[sample_idx]

    # Verify the selected image belongs to the correct class
    assert int(label) == class_idx, (
        f"Incorrect image selected for {class_name}"
    )

    original_image = image.clone()

    # Display original image
    axes[class_idx, 0].imshow(
        image.squeeze(),
        cmap="gray",
        vmin=0,
        vmax=1
    )

    axes[class_idx, 0].set_title(
        f"{class_name}\nOriginal",
        fontsize=10
    )

    axes[class_idx, 0].axis("off")

    # Display Medium occlusion at three positions
    for col, position in enumerate(
        ["Top", "Center", "Bottom"], start=1
    ):

        occluded, percentage = apply_occlusion(
            image,
            position,
            occlusion_sizes["Medium"]
        )

        axes[class_idx, col].imshow(
            occluded.squeeze(),
            cmap="gray",
            vmin=0,
            vmax=1
        )

        axes[class_idx, col].set_title(
            f"{position}\nHidden: {percentage:.1f}%",
            fontsize=10
        )

        axes[class_idx, col].axis("off")

    # Verify all nine occlusion settings for this class
    for position in occlusion_positions:
        for size in occlusion_sizes.values():

            occluded, percentage = apply_occlusion(
                image, position, size
            )

            assert occluded.shape == image.shape
            assert 0 <= percentage <= 100
            assert torch.equal(image, original_image)

            verified_cases += 1

plt.tight_layout(h_pad=2.0)
plt.show()

print("Classes checked:", len(classes))
print("Occlusion cases verified:", verified_cases)
print("No retraining performed.")

In [ ]:
# Cell 18: Consolidate Occlusion Evaluation and Results

import os
import numpy as np
import pandas as pd
import torch

results_dir = os.path.join(dataset_path, "results")
os.makedirs(results_dir, exist_ok=True)

detail_path = os.path.join(
    results_dir, "per_image_occlusion_results.csv"
)
summary_path = os.path.join(
    results_dir, "occlusion_summary.csv"
)
classwise_path = os.path.join(
    results_dir, "occlusion_classwise.csv"
)

# Load existing detailed results if available
if os.path.exists(detail_path):
    detailed_df = pd.read_csv(detail_path)
    print("Loaded existing per-image results.")

else:
    # Restore the trained model without retraining
    if "FashionCNN" not in globals():
        raise RuntimeError(
            "Run Cell 9 to define FashionCNN. Do not run Cell 11."
        )

    eval_model = FashionCNN().to(device)
    eval_model.load_state_dict(
        torch.load(
            os.path.join(dataset_path, "best_fashion_cnn.pth"),
            map_location=device,
            weights_only=True
        )
    )
    eval_model.eval()

    baseline_data = pd.read_csv(
        os.path.join(results_dir, "baseline_predictions.csv")
    ).sort_values("image_index").reset_index(drop=True)

    records = []

    # Evaluate every occlusion position and size
    with torch.inference_mode():
        for position in occlusion_positions:
            for size_name, size_value in occlusion_sizes.items():

                image_index = 0

                for images, labels in test_loader:
                    masked_images = []
                    percentages = []

                    for image in images:
                        masked, pct = apply_occlusion(
                            image, position, size_value
                        )
                        masked_images.append(masked)
                        percentages.append(pct)

                    predictions = eval_model(
                        torch.stack(masked_images).to(device)
                    ).argmax(1).cpu().tolist()

                    for i, true_label in enumerate(labels.tolist()):
                        original = baseline_data.iloc[image_index]

                        assert int(original["true_label"]) == true_label

                        records.append({
                            "image_index": image_index,
                            "class": classes[true_label],
                            "position": position,
                            "size": size_name,
                            "hidden_foreground_pct": percentages[i],
                            "original_correct": int(
                                original["original_prediction"] == true_label
                            ),
                            "occluded_correct": int(
                                predictions[i] == true_label
                            ),
                            "occluded_prediction": predictions[i]
                        })

                        image_index += 1

                print(f"Evaluated: {position} - {size_name}")

    detailed_df = pd.DataFrame(records)
    detailed_df.to_csv(detail_path, index=False)

# Verify completeness of the detailed results
assert len(detailed_df) == 90000
assert not detailed_df.duplicated(
    ["image_index", "position", "size"]
).any()
assert detailed_df.groupby("image_index").size().eq(9).all()

# Calculate the original test accuracy
baseline_accuracy = (
    detailed_df.drop_duplicates("image_index")[
        "original_correct"
    ].mean() * 100
)

# Create overall occlusion summary
occlusion_summary_df = detailed_df.groupby(
    ["position", "size"], as_index=False
).agg(
    overall_accuracy=("occluded_correct", "mean"),
    mean_hidden_foreground_pct=("hidden_foreground_pct", "mean")
)

occlusion_summary_df["overall_accuracy"] *= 100
occlusion_summary_df["accuracy_drop"] = (
    baseline_accuracy - occlusion_summary_df["overall_accuracy"]
)

occlusion_summary_df = occlusion_summary_df[
    ["position", "size", "overall_accuracy",
     "accuracy_drop", "mean_hidden_foreground_pct"]
]

# Create class-wise summary
classwise_summary_df = detailed_df.groupby(
    ["position", "size", "class"], as_index=False
).agg(
    class_accuracy=("occluded_correct", "mean"),
    baseline_class_accuracy=("original_correct", "mean")
)

classwise_summary_df["class_accuracy_drop"] = (
    classwise_summary_df["baseline_class_accuracy"]
    - classwise_summary_df["class_accuracy"]
) * 100

classwise_summary_df["class_accuracy"] *= 100

classwise_summary_df = classwise_summary_df[
    ["position", "size", "class",
     "class_accuracy", "class_accuracy_drop"]
]

# Compare recomputed results with existing saved results
def verify_existing_results(new_df, path, keys, metrics):
    if os.path.exists(path):
        previous = pd.read_csv(path)

        check = new_df.merge(
            previous,
            on=keys,
            suffixes=("_new", "_old"),
            validate="one_to_one"
        )

        assert len(check) == len(new_df) == len(previous)

        for metric in metrics:
            assert np.allclose(
                check[f"{metric}_new"],
                check[f"{metric}_old"],
                atol=1e-6,
                rtol=0
            ), f"Results changed: {metric}"

        print(f"Verified unchanged: {os.path.basename(path)}")

verify_existing_results(
    occlusion_summary_df,
    summary_path,
    ["position", "size"],
    ["overall_accuracy", "accuracy_drop",
     "mean_hidden_foreground_pct"]
)

verify_existing_results(
    classwise_summary_df,
    classwise_path,
    ["position", "size", "class"],
    ["class_accuracy", "class_accuracy_drop"]
)

# Save verified results
occlusion_summary_df.to_csv(summary_path, index=False)
classwise_summary_df.to_csv(classwise_path, index=False)

print("\nBaseline Accuracy:", round(baseline_accuracy, 2), "%")
print("Overall conditions:", len(occlusion_summary_df))
print("Class-wise results:", len(classwise_summary_df))
print("Per-image records:", len(detailed_df))
print("All results consolidated successfully!")

In [ ]:
# Cell 19: Verify Saved Occlusion Results Before Cleanup

import os
import pandas as pd

results_path = os.path.join(dataset_path, "results")

files = {
    "Occlusion Summary": ("occlusion_summary.csv", 9),
    "Class-Wise Results": ("occlusion_classwise.csv", 90),
    "Per-Image Results": ("per_image_occlusion_results.csv", 90000)
}

for name, (filename, expected_rows) in files.items():

    path = os.path.join(results_path, filename)

    assert os.path.exists(path), f"Missing file: {filename}"

    data = pd.read_csv(path)

    assert len(data) == expected_rows, (
        f"Unexpected number of rows in {filename}"
    )

    print(f"{name}: {len(data)} rows - Saved correctly")

print("\nAll occlusion result files are available!")

In [ ]:
# Cell 20: Analyze Class-Wise Accuracy Drop

import pandas as pd
import os
from IPython.display import display

# Load saved class-wise occlusion results
classwise_path = os.path.join(
    dataset_path, "results", "occlusion_classwise.csv"
)

classwise_results = pd.read_csv(classwise_path)

# Create a label for each occlusion condition
classwise_results["condition"] = (
    classwise_results["position"] + " - " +
    classwise_results["size"]
)

# Define the order of the nine conditions
conditions = [
    f"{position} - {size}"
    for position in ["Top", "Center", "Bottom"]
    for size in ["Small", "Medium", "Large"]
]

# Compare accuracy drops across the 10 classes
classwise_table = classwise_results.pivot(
    index="class",
    columns="condition",
    values="class_accuracy_drop"
)

# Organize rows and columns
classwise_table = classwise_table.reindex(
    index=classes,
    columns=conditions
)

print("Class-Wise Accuracy Drop (percentage points):")
display(classwise_table.round(2))

print("\nNumber of clothing classes:", len(classwise_table))
print("Number of occlusion conditions:", len(conditions))

In [ ]:
# Cell 21: Visualize Class-Wise Accuracy Drop Using Heatmap

import matplotlib.pyplot as plt
import numpy as np
import os

# Prepare class-wise accuracy drop values
heatmap_data = classwise_table.values

# Create the heatmap
fig, ax = plt.subplots(figsize=(16, 8))

im = ax.imshow(
    heatmap_data,
    cmap="coolwarm",
    aspect="auto"
)

# Set axis labels
ax.set_xticks(range(len(conditions)))
ax.set_xticklabels(conditions, rotation=45, ha="right")

ax.set_yticks(range(len(classes)))
ax.set_yticklabels(classes)

# Add values inside each cell
for i in range(len(classes)):
    for j in range(len(conditions)):
        ax.text(
            j, i,
            f"{heatmap_data[i, j]:.1f}",
            ha="center",
            va="center",
            fontsize=8
        )

# Add title and colorbar
ax.set_title(
    "Class-Wise Accuracy Drop Under Occlusion",
    fontsize=14
)

ax.set_xlabel("Occlusion Condition")
ax.set_ylabel("Clothing Class")

cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Accuracy Drop (Percentage Points)")

plt.tight_layout()

# Save the figure
heatmap_path = os.path.join(
    dataset_path,
    "results",
    "classwise_heatmap.png"
)

plt.savefig(heatmap_path, dpi=300, bbox_inches="tight")
plt.show()

print("Heatmap saved to:", heatmap_path)

In [ ]:
# Cell 22: Analyze Accuracy Drop vs. Hidden Foreground Percentage

import pandas as pd
import matplotlib.pyplot as plt
import os

# Load saved occlusion results
results_path = os.path.join(
    dataset_path, "results", "occlusion_summary.csv"
)

results = pd.read_csv(results_path)

# Create the scatter plot
plt.figure(figsize=(11, 7))

plt.scatter(
    results["mean_hidden_foreground_pct"],
    results["accuracy_drop"],
    s=100
)

# Label each occlusion condition
for _, row in results.iterrows():

    label = f"{row['position']} - {row['size']}"

    plt.annotate(
        label,
        (
            row["mean_hidden_foreground_pct"],
            row["accuracy_drop"]
        ),
        xytext=(5, 7),
        textcoords="offset points",
        fontsize=9
    )

# Add titles and labels
plt.title(
    "Accuracy Drop vs. Hidden Foreground Percentage",
    fontsize=14
)

plt.xlabel("Mean Hidden Foreground (%)")
plt.ylabel("Accuracy Drop (Percentage Points)")

plt.grid(True, alpha=0.3)
plt.tight_layout()

# Save the figure
figure_path = os.path.join(
    dataset_path,
    "results",
    "foreground_vs_accuracy_drop.png"
)

plt.savefig(figure_path, dpi=300, bbox_inches="tight")
plt.show()

print("Figure saved to:", figure_path)

In [ ]:
# Cell 23: Validate Per-Image Occlusion Results

import pandas as pd
import os

# Load detailed results
detailed_path = os.path.join(
    dataset_path, "results", "per_image_occlusion_results.csv"
)

df = pd.read_csv(detailed_path)

# Verify dataset completeness
assert len(df) == 90000
assert not df.isnull().any().any()

# Verify no duplicate experiments
assert not df.duplicated(
    ["image_index", "position", "size"]
).any()

# Verify each image has 9 conditions
assert df.groupby("image_index").size().eq(9).all()

# Verify valid foreground percentages
assert df["hidden_foreground_pct"].between(0, 100).all()

# Calculate summary statistics
audit = df.groupby(
    ["position", "size"], as_index=False
).agg(
    images=("image_index", "count"),
    mean_hidden=("hidden_foreground_pct", "mean"),
    median_hidden=("hidden_foreground_pct", "median"),
    min_hidden=("hidden_foreground_pct", "min"),
    accuracy=("occluded_correct", "mean")
)

# Convert accuracy to percentage
audit["accuracy"] *= 100

print("Total records:", len(df))
print("Duplicate records: 0")
print("All images have 9 conditions: True")

print("\nOcclusion Quality Report:")
display(audit.round(2))

# Check masks that cover no foreground pixels
zero_masks = (df["hidden_foreground_pct"] == 0).sum()

print("\nZero-foreground occlusions:", zero_masks)

In [ ]:
# Cell 24: Compare Occlusion Positions at Similar Foreground Coverage

import pandas as pd
import os

# Use the saved per-image evaluation results
df = pd.read_csv(
    os.path.join(
        dataset_path,
        "results",
        "per_image_occlusion_results.csv"
    )
)

# Define a fixed coverage tolerance
tolerance = 3.0

positions = ["Top", "Center", "Bottom"]
sizes = ["Small", "Medium", "Large"]

# Organize results for the same image and mask size
paired = df.pivot(
    index=["image_index", "class", "size"],
    columns="position",
    values=[
        "hidden_foreground_pct",
        "occluded_correct",
        "original_correct"
    ]
)

# Extract foreground coverage and classification outcomes
coverage = paired["hidden_foreground_pct"][positions]
correct = paired["occluded_correct"][positions]
original = paired["original_correct"]["Top"]

# Calculate coverage difference among the three positions
coverage_difference = (
    coverage.max(axis=1) - coverage.min(axis=1)
)

matched = coverage_difference <= tolerance

comparison_results = []

# Compare positions separately for each mask size
for size in sizes:

    size_mask = coverage.index.get_level_values("size") == size
    selected = size_mask & matched

    n = int(selected.sum())

    if n == 0:
        print(f"{size}: No comparable images found.")
        continue

    baseline_acc = original[selected].mean() * 100

    for position in positions:

        position_acc = correct.loc[
            selected, position
        ].mean() * 100

        comparison_results.append({
            "size": size,
            "position": position,
            "matched_images": n,
            "baseline_accuracy": baseline_acc,
            "occluded_accuracy": position_acc,
            "accuracy_drop": baseline_acc - position_acc
        })

    print(f"{size}: {n} comparable images out of 10000")

# Display comparison results
matched_df = pd.DataFrame(comparison_results)

print("\nMatched Position Comparison:")
display(matched_df.round(2))

# Check the number of matched images in each class
matched_counts = coverage.index.to_frame(index=False)
matched_counts["matched"] = matched.to_numpy()

class_counts = matched_counts.groupby(
    ["size", "class"]
)["matched"].sum().unstack(fill_value=0)

print("\nComparable Images by Clothing Class:")
display(class_counts.reindex(sizes))

# Save results
save_path = os.path.join(
    dataset_path,
    "results",
    "matched_position_comparison.csv"
)

matched_df.to_csv(save_path, index=False)

print("\nResults saved to:", save_path)

In [ ]:
# Cell 25: Analyze Matched Occlusion Results by Clothing Class

import pandas as pd
import numpy as np
import os
from IPython.display import display

# Load saved per-image results
df = pd.read_csv(
    os.path.join(
        dataset_path, "results",
        "per_image_occlusion_results.csv"
    )
)

positions = ["Top", "Center", "Bottom"]
sizes = ["Small", "Medium", "Large"]

# Organize results by image, class, and size
wide = df.pivot(
    index=["image_index", "class", "size"],
    columns="position",
    values=["hidden_foreground_pct", "occluded_correct"]
)

coverage = wide["hidden_foreground_pct"][positions]
correct = wide["occluded_correct"][positions]

# Keep images with similar foreground coverage
tolerance = 3.0

matched = (
    coverage.max(axis=1) - coverage.min(axis=1)
) <= tolerance

results = []

# Analyze each clothing class separately
for size in sizes:
    for class_name in classes:

        selected = (
            matched
            & (coverage.index.get_level_values("size") == size)
            & (coverage.index.get_level_values("class") == class_name)
        )

        n = int(selected.sum())

        if n > 0:
            accuracy = correct.loc[
                selected, positions
            ].mean() * 100
        else:
            accuracy = pd.Series(np.nan, index=positions)

        results.append({
            "Size": size,
            "Class": class_name,
            "Matched Images": n,
            "Top Accuracy": accuracy["Top"],
            "Center Accuracy": accuracy["Center"],
            "Bottom Accuracy": accuracy["Bottom"],
            "Center Extra Drop vs Top": (
                accuracy["Top"] - accuracy["Center"]
            ),
            "Sample Status": (
                "Limited" if n < 100 else "Exploratory"
            )
        })

# Create results table
class_matched_df = pd.DataFrame(results)

# Display a separate table for each mask size
for size in sizes:
    print(f"\n{size} Occlusion - Class-Wise Comparison")

    display(
        class_matched_df[
            class_matched_df["Size"] == size
        ].round(2)
    )

# Save results to Google Drive
save_path = os.path.join(
    dataset_path,
    "results",
    "classwise_matched_comparison.csv"
)

class_matched_df.to_csv(save_path, index=False)

print("\nResults saved to:", save_path)

In [ ]:
# Cell 26: Consolidate Fixed-Center and Foreground-Aware Comparisons

import os
import torch
import numpy as np
import pandas as pd
from IPython.display import display

results_dir = os.path.join(dataset_path, "results")

paired_path = os.path.join(
    results_dir, "fixed_vs_foreground_per_image.csv"
)
summary_path = os.path.join(
    results_dir, "fixed_vs_foreground_comparison.csv"
)
classwise_path = os.path.join(
    results_dir, "classwise_method_comparison.csv"
)
fg_path = os.path.join(
    results_dir, "per_image_occlusion_results.csv"
)

sizes = ["Small", "Medium", "Large"]

# Step 1: Load existing results or reproduce the evaluation
if os.path.exists(paired_path):

    comparison = pd.read_csv(paired_path)
    print("Loaded existing per-image comparison results.")

else:

    # Load saved CNN weights without retraining
    if "FashionCNN" not in globals():
        raise RuntimeError(
            "Run Cell 9 to define CNN, but do not run Cell 11."
        )

    eval_model = FashionCNN().to(device)

    eval_model.load_state_dict(
        torch.load(
            os.path.join(dataset_path, "best_fashion_cnn.pth"),
            map_location=device,
            weights_only=True
        )
    )

    eval_model.eval()

    fixed_records = []

    # Evaluate fixed-center occlusion for each mask size
    with torch.inference_mode():

        for size_name, size_value in occlusion_sizes.items():

            start = (28 - size_value) // 2
            end = start + size_value
            image_index = 0

            for images, labels in test_loader:

                foreground = images[:, 0] > foreground_threshold

                total_fg = foreground.sum(dim=(1, 2))
                hidden_fg = foreground[
                    :, start:end, start:end
                ].sum(dim=(1, 2))

                hidden_pct = (
                    100 * hidden_fg.float() /
                    total_fg.clamp(min=1)
                )

                occluded = images.clone()
                occluded[:, :, start:end, start:end] = 0.0

                predictions = eval_model(
                    occluded.to(device)
                ).argmax(dim=1).cpu().tolist()

                for i, label in enumerate(labels.tolist()):

                    fixed_records.append({
                        "image_index": image_index,
                        "class": classes[label],
                        "size": size_name,
                        "fixed_correct": int(
                            predictions[i] == label
                        ),
                        "fixed_hidden_pct": float(hidden_pct[i]),
                        "fixed_zero_mask": int(hidden_fg[i] == 0)
                    })

                    image_index += 1

            assert image_index == 10000
            print(f"Evaluated Fixed-Center: {size_name}")

    fixed_df = pd.DataFrame(fixed_records)

    # Match the same images with our Foreground-Aware results
    fg_df = pd.read_csv(fg_path)

    fg_center = fg_df[
        fg_df["position"] == "Center"
    ][[
        "image_index", "class", "size",
        "hidden_foreground_pct", "occluded_correct"
    ]].rename(columns={
        "hidden_foreground_pct": "fg_hidden_pct",
        "occluded_correct": "fg_correct"
    })

    comparison = fixed_df.merge(
        fg_center,
        on=["image_index", "class", "size"],
        validate="one_to_one"
    )

# Step 2: Verify detailed comparison data
assert len(comparison) == 30000

assert not comparison.duplicated(
    ["image_index", "size"]
).any()

assert comparison.groupby("size").size().eq(10000).all()

assert not comparison.isnull().any().any()

# Step 3: Calculate overall comparison
summary = comparison.groupby(
    "size", as_index=False
).agg(
    fixed_accuracy=("fixed_correct", "mean"),
    fg_accuracy=("fg_correct", "mean"),
    fixed_hidden=("fixed_hidden_pct", "mean"),
    fg_hidden=("fg_hidden_pct", "mean"),
    fixed_zero_masks=("fixed_zero_mask", "sum")
)

summary["fixed_accuracy"] *= 100
summary["fg_accuracy"] *= 100

overall_comparison = summary.rename(columns={
    "size": "Size",
    "fixed_accuracy": "Fixed Accuracy",
    "fg_accuracy": "Foreground-Aware Accuracy",
    "fixed_hidden": "Fixed Hidden FG (%)",
    "fg_hidden": "Foreground-Aware Hidden FG (%)",
    "fixed_zero_masks": "Fixed Zero-FG Masks"
})

# Step 4: Calculate class-wise comparison
class_comparison = comparison.groupby(
    ["size", "class"], as_index=False
).agg(
    fixed_accuracy=("fixed_correct", "mean"),
    fg_accuracy=("fg_correct", "mean"),
    fixed_hidden=("fixed_hidden_pct", "mean"),
    fg_hidden=("fg_hidden_pct", "mean"),
    fixed_zero_masks=("fixed_zero_mask", "sum")
)

class_comparison["fixed_accuracy"] *= 100
class_comparison["fg_accuracy"] *= 100

class_comparison["accuracy_difference"] = (
    class_comparison["fg_accuracy"]
    - class_comparison["fixed_accuracy"]
)

# Step 5: Verify that saved results remain unchanged
def verify_results(new_df, path, keys, metrics):

    if os.path.exists(path):

        old_df = pd.read_csv(path)

        merged = new_df.merge(
            old_df,
            on=keys,
            suffixes=("_new", "_old"),
            validate="one_to_one"
        )

        assert len(merged) == len(new_df) == len(old_df)

        for metric in metrics:
            assert np.allclose(
                merged[f"{metric}_new"],
                merged[f"{metric}_old"],
                atol=1e-3,
                rtol=0
            ), f"Results changed: {metric}"

        print("Verified unchanged:", os.path.basename(path))

verify_results(
    overall_comparison,
    summary_path,
    ["Size"],
    [
        "Fixed Accuracy",
        "Foreground-Aware Accuracy",
        "Fixed Hidden FG (%)",
        "Foreground-Aware Hidden FG (%)",
        "Fixed Zero-FG Masks"
    ]
)

verify_results(
    class_comparison,
    classwise_path,
    ["size", "class"],
    [
        "fixed_accuracy",
        "fg_accuracy",
        "fixed_hidden",
        "fg_hidden",
        "fixed_zero_masks",
        "accuracy_difference"
    ]
)

# Step 6: Save verified results
comparison.to_csv(paired_path, index=False)
overall_comparison.to_csv(summary_path, index=False)
class_comparison.to_csv(classwise_path, index=False)

print("\nOverall Comparison:")
display(overall_comparison.round(2))

print("\nClass-Wise Results:")
for size in sizes:
    print(f"\n{size}:")
    display(
        class_comparison[
            class_comparison["size"] == size
        ].round(2)
    )

print("\nOverall conditions:", len(overall_comparison))
print("Class-wise results:", len(class_comparison))
print("Per-image records:", len(comparison))
print("Comparison consolidation completed successfully!")

In [ ]:
# Cell 27: Calculate Paired Bootstrap Confidence Intervals

import numpy as np
import pandas as pd
import os
from IPython.display import display

# Save per-image comparison results for reproducibility
paired_path = os.path.join(
    dataset_path,
    "results",
    "fixed_vs_foreground_per_image.csv"
)

comparison.to_csv(paired_path, index=False)

# Set bootstrap parameters
rng = np.random.default_rng(42)
n_bootstrap = 1000

bootstrap_results = []

for size in ["Small", "Medium", "Large"]:
    for class_name in classes:

        # Select the same images evaluated by both methods
        subset = comparison[
            (comparison["size"] == size) &
            (comparison["class"] == class_name)
        ]

        fixed = subset["fixed_correct"].to_numpy()
        fg = subset["fg_correct"].to_numpy()

        # Difference in correctness for each image
        differences = fg - fixed

        # Resample paired image outcomes
        indices = rng.integers(
            0, len(differences),
            size=(n_bootstrap, len(differences))
        )

        bootstrap_differences = (
            differences[indices].mean(axis=1) * 100
        )

        # Calculate 95% confidence interval
        lower, upper = np.percentile(
            bootstrap_differences, [2.5, 97.5]
        )

        bootstrap_results.append({
            "Size": size,
            "Class": class_name,
            "Images": len(subset),
            "Accuracy Difference": differences.mean() * 100,
            "CI Lower": lower,
            "CI Upper": upper
        })

# Create and display results
bootstrap_df = pd.DataFrame(bootstrap_results)

for size in ["Small", "Medium", "Large"]:

    print(f"\n{size} - Bootstrap Confidence Intervals")

    display(
        bootstrap_df[
            bootstrap_df["Size"] == size
        ].round(2)
    )

# Save results
save_path = os.path.join(
    dataset_path,
    "results",
    "bootstrap_confidence_intervals.csv"
)

bootstrap_df.to_csv(save_path, index=False)

print("\nBootstrap results saved to:", save_path)
print("Per-image comparison saved to:", paired_path)

In [ ]:
# Cell 28: Visualize Bootstrap Confidence Intervals

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# Load saved bootstrap results
bootstrap_path = os.path.join(
    dataset_path,
    "results",
    "bootstrap_confidence_intervals.csv"
)

bootstrap_results = pd.read_csv(bootstrap_path)

# Create a separate plot for each occlusion size
for size in ["Small", "Medium", "Large"]:

    data = bootstrap_results[
        bootstrap_results["Size"] == size
    ].set_index("Class").reindex(classes)

    differences = data["Accuracy Difference"].to_numpy()
    lower = data["CI Lower"].to_numpy()
    upper = data["CI Upper"].to_numpy()

    # Calculate confidence interval error bars
    errors = np.array([
        differences - lower,
        upper - differences
    ])

    fig, ax = plt.subplots(figsize=(10, 6))

    ax.errorbar(
        differences,
        range(len(classes)),
        xerr=errors,
        fmt="o",
        capsize=4
    )

    # Zero indicates equal accuracy
    ax.axvline(0, linestyle="--", linewidth=1)

    ax.set_yticks(range(len(classes)))
    ax.set_yticklabels(classes)
    ax.invert_yaxis()

    ax.set_xlabel("Accuracy Difference (Percentage Points)")
    ax.set_title(
        f"{size} Occlusion: Accuracy Difference with 95% CI"
    )

    ax.grid(axis="x", alpha=0.3)

    plt.tight_layout()

    # Save figure to Google Drive
    save_path = os.path.join(
        dataset_path,
        "results",
        f"bootstrap_ci_{size.lower()}.png"
    )

    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

    print(f"{size} figure saved:", save_path)

In [ ]:
# Cell 29: Plot Training and Validation Curves

import os
import pandas as pd
import matplotlib.pyplot as plt

# Class names (same order as the dataset labels)
classes = [
    "T-shirt/top", "Trouser", "Pullover", "Dress",
    "Coat", "Sandal", "Shirt", "Sneaker",
    "Bag", "Ankle boot"
]

# Save a figure to Drive; if Drive is read-only, save to Colab storage
def save_fig(name):
    for folder in [os.path.join(dataset_path, "results"), "/content"]:
        try:
            path = os.path.join(folder, name)
            plt.savefig(path, dpi=300, bbox_inches="tight")
            print("Saved to:", path)
            return
        except (PermissionError, OSError):
            continue
    print("Could not save the figure.")

# Load the saved training history
history_df = pd.read_csv(
    os.path.join(dataset_path, "results", "training_history.csv")
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Loss curves
axes[0].plot(history_df["epoch"], history_df["train_loss"], label="Train")
axes[0].plot(history_df["epoch"], history_df["val_loss"], label="Validation")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss per Epoch")
axes[0].legend()

# Accuracy curves
axes[1].plot(history_df["epoch"], history_df["train_accuracy"], label="Train")
axes[1].plot(history_df["epoch"], history_df["val_accuracy"], label="Validation")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy (%)")
axes[1].set_title("Accuracy per Epoch")
axes[1].legend()

plt.tight_layout()
save_fig("training_curves.png")
plt.show()

print("Final epoch:")
print(history_df.iloc[-1].round(4))

In [ ]:
# Cell 30: Confusion Matrix and Classification Report (Clean Test Images)

import os
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

# Load the saved baseline predictions
baseline = pd.read_csv(
    os.path.join(dataset_path, "results", "baseline_predictions.csv")
)

# Verify the baseline accuracy
accuracy = (baseline["true_label"] == baseline["original_prediction"]).mean() * 100
print(f"Baseline Test Accuracy: {accuracy:.2f}%")

# Confusion matrix
cm = confusion_matrix(baseline["true_label"], baseline["original_prediction"])

fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay(cm, display_labels=classes).plot(
    ax=ax, xticks_rotation=45, cmap="Blues", values_format="d"
)
ax.set_title("Confusion Matrix on Clean Test Images")

plt.tight_layout()
save_fig("confusion_matrix_clean.png")
plt.show()

# Precision, recall, and F1-score for each class
print(classification_report(
    baseline["true_label"],
    baseline["original_prediction"],
    target_names=classes,
    digits=4
))

In [ ]:
# Cell 31: Analyze Class Substitutions Under Large Center Occlusion

import os
import pandas as pd

# Load the saved per-image occlusion results
detailed = pd.read_csv(
    os.path.join(dataset_path, "results", "per_image_occlusion_results.csv")
)

selected = detailed[
    (detailed["position"] == "Center") & (detailed["size"] == "Large")
]

# Show the three most frequent predictions for the most affected classes
for class_name in ["Sneaker", "Coat", "Ankle boot"]:

    predictions = selected[
        selected["class"] == class_name
    ]["occluded_prediction"].map(lambda i: classes[i])

    top3 = predictions.value_counts(normalize=True).head(3) * 100

    print(f"\n{class_name} (Center - Large) is predicted as:")
    for label, percent in top3.items():
        print(f"  {label}: {percent:.1f}%")

In [ ]:
# Cell 32: Inspect Saved Per-Image Occlusion Results

import pandas as pd
import os

file_path = os.path.join(dataset_path, "results", "per_image_occlusion_results.csv")

assert os.path.exists(file_path), "Results file not found!"

df = pd.read_csv(file_path)

print("Number of records:", len(df))
print("Columns:", df.columns.tolist())

display(df.head(3))


In [ ]:
# Cell 33: Paired Bootstrap 95% Confidence Intervals

import numpy as np
import pandas as pd
from pathlib import Path

# 1. Load previously saved results
results_dir = Path(dataset_path) / "results"
input_file = results_dir / "per_image_occlusion_results.csv"

data = pd.read_csv(input_file)

# 2. Bootstrap settings
N_BOOTSTRAP = 1000
SEED = 42
rng = np.random.default_rng(SEED)

results = []

positions = ["Top", "Center", "Bottom"]
sizes = ["Small", "Medium", "Large"]

# 3. Calculate paired bootstrap CI for each condition
for position in positions:
    for size in sizes:

        subset = data[
            (data["position"] == position) &
            (data["size"] == size)
        ].sort_values("image_index")

        assert len(subset) == 10000, (
            f"Unexpected image count: {position}, {size}"
        )

        assert subset["image_index"].is_unique, (
            f"Duplicate image indexes: {position}, {size}"
        )

        original = subset["original_correct"].to_numpy()
        occluded = subset["occluded_correct"].to_numpy()

        # Paired accuracy drop, in percentage points
        differences = (original - occluded) * 100
        accuracy_drop = differences.mean()

        # Resample image pairs with replacement
        bootstrap_drops = np.empty(N_BOOTSTRAP)

        for start in range(0, N_BOOTSTRAP, 100):
            batch_size = min(100, N_BOOTSTRAP - start)

            indices = rng.integers(
                0,
                len(differences),
                size=(batch_size, len(differences))
            )

            bootstrap_drops[start:start + batch_size] = (
                differences[indices].mean(axis=1)
            )

        # 95% percentile confidence interval
        ci_lower, ci_upper = np.percentile(
            bootstrap_drops, [2.5, 97.5]
        )

        results.append({
            "Position": position,
            "Size": size,
            "Accuracy_Drop": accuracy_drop,
            "CI_95_Lower": ci_lower,
            "CI_95_Upper": ci_upper
        })

# 4. Create results table
bootstrap_df = pd.DataFrame(results)

print("Bootstrap 95% Confidence Intervals")
display(bootstrap_df.round(2))

# 5. Save new results without modifying old files
output_file = results_dir / "bootstrap_overall_accuracy_drop.csv"
bootstrap_df.to_csv(output_file, index=False)

print("\nSaved to:", output_file)


In [ ]:

# Cell 34: Visualize Bootstrap 95% Confidence Intervals

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# 1. Load saved Bootstrap results
results_dir = Path(dataset_path) / "results"

ci_df = pd.read_csv(
    results_dir / "bootstrap_overall_accuracy_drop.csv"
)

# 2. Define positions and mask sizes
positions = ["Top", "Center", "Bottom"]
sizes = ["Small", "Medium", "Large"]

x = np.arange(len(sizes))

# 3. Create the figure
fig, ax = plt.subplots(figsize=(9, 5.5))

for position in positions:

    subset = (
        ci_df[ci_df["Position"] == position]
        .set_index("Size")
        .loc[sizes]
    )

    drops = subset["Accuracy_Drop"].to_numpy()

    lower_errors = (
        drops - subset["CI_95_Lower"].to_numpy()
    )

    upper_errors = (
        subset["CI_95_Upper"].to_numpy() - drops
    )

    ax.errorbar(
        x,
        drops,
        yerr=[lower_errors, upper_errors],
        marker="o",
        capsize=5,
        linewidth=2,
        markersize=7,
        label=position
    )

# 4. Format the chart
ax.set_xticks(x)
ax.set_xticklabels(sizes)

ax.set_xlabel("Occlusion Mask Size")
ax.set_ylabel("Accuracy Drop (Percentage Points)")

ax.set_title(
    "CNN Accuracy Drop Under Occlusion\n"
    "with Bootstrap 95% Confidence Intervals"
)

ax.legend(title="Mask Position")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()

# 5. Save the figure
output_file = (
    results_dir / "bootstrap_accuracy_drop_ci.png"
)

fig.savefig(output_file, dpi=300, bbox_inches="tight")

plt.show()

print("Figure saved to:", output_file)


In [ ]:
import pandas as pd, numpy as np, glob, os
for f in sorted(glob.glob("reference_results/*.csv")):
    new = os.path.join("results", os.path.basename(f))
    if not os.path.exists(new):
        print("MISSING", os.path.basename(f)); continue
    a, b = pd.read_csv(f), pd.read_csv(new)
    ok = a.shape == b.shape and np.allclose(a.select_dtypes("number"), b.select_dtypes("number"), atol=1e-9)
    print(("OK   " if ok else "DIFF ") + os.path.basename(f))